# Preprocessing

In [3]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from pathlib import Path
from src.aggregate import aggregate

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")

`365243` in `DAYS_*` = NA (per competition host).

In [4]:
prev = pd.read_csv(RAW / "previous_application.csv")
day_cols = ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION"]
prev[day_cols] = prev[day_cols].replace(365243, np.nan)
for c in ["NAME_GOODS_CATEGORY", "NAME_CASH_LOAN_PURPOSE", "PRODUCT_COMBINATION"]:
    keep = prev[c].value_counts().head(8).index
    prev[c] = prev[c].where(prev[c].isin(keep), "Other")
prev["credit_goods_ratio"] = prev["AMT_CREDIT"] / prev["AMT_GOODS_PRICE"]
prev["credit_goods_diff"] = prev["AMT_CREDIT"] - prev["AMT_GOODS_PRICE"]
prev["application_credit_ratio"] = prev["AMT_APPLICATION"] / prev["AMT_CREDIT"]
prev["application_credit_diff"] = prev["AMT_APPLICATION"] - prev["AMT_CREDIT"]
prev["approved"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype("int8")
prev["refused"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype("int8")
prev = prev.replace([np.inf, -np.inf], np.nan)
prev.shape

(1670214, 43)

# Feature Engineering

## Iteration 1

In [5]:
cat = ["NAME_CONTRACT_TYPE", "NAME_CONTRACT_STATUS", "NAME_CLIENT_TYPE", "NAME_PORTFOLIO",
       "NAME_PRODUCT_TYPE", "NAME_YIELD_GROUP", "CODE_REJECT_REASON", "NAME_PAYMENT_TYPE",
       "NAME_GOODS_CATEGORY", "NAME_CASH_LOAN_PURPOSE", "PRODUCT_COMBINATION"]
p = aggregate(prev, "SK_ID_CURR", "prev", cat_cols=cat, ignore=["SK_ID_PREV"])
p["prev_credit_to_application"] = p["prev_AMT_CREDIT_sum"] / p["prev_AMT_APPLICATION_sum"]
p["prev_down_payment_ratio"] = p["prev_AMT_DOWN_PAYMENT_sum"] / p["prev_AMT_CREDIT_sum"]
p["prev_annuity_credit"] = p["prev_AMT_ANNUITY_sum"] / p["prev_AMT_CREDIT_sum"]
p = p.replace([np.inf, -np.inf], np.nan)

In [6]:
disb = pd.DataFrame({"SK_ID_CURR": prev["SK_ID_CURR"].to_numpy(),
                     "disbursed": prev["DAYS_FIRST_DUE"].notna().to_numpy("int8")})
p = p.join(disb.groupby("SK_ID_CURR")["disbursed"].mean().rename("prev_disbursed_share"))
p.shape

(338857, 256)

In [7]:
prev = prev.sort_values(["SK_ID_CURR", "DAYS_DECISION"], ascending=[True, False])
prev["rank"] = prev.groupby("SK_ID_CURR").cumcount()
parts = []
for N in [1, 3, 5]:
    r = prev[prev["rank"] < N].groupby("SK_ID_CURR")
    parts += [r["approved"].mean().rename(f"prev_approved_last{N}"),
              r["refused"].mean().rename(f"prev_refused_last{N}"),
              r["CNT_PAYMENT"].mean().rename(f"prev_term_last{N}"),
              r["DAYS_DECISION"].mean().rename(f"prev_days_decision_last{N}"),
              r["AMT_CREDIT"].mean().rename(f"prev_credit_last{N}"),
              r["application_credit_ratio"].mean().rename(f"prev_app_credit_ratio_last{N}")]
p = p.join(pd.concat(parts, axis=1))
p.shape

(338857, 274)

## Iteration 2

In [8]:
for status, tag in [("Approved", "appr"), ("Refused", "ref")]:
    sub = prev[prev["NAME_CONTRACT_STATUS"] == status]
    a = aggregate(sub, "SK_ID_CURR", f"prev_{tag}",
                  cat_cols=["NAME_YIELD_GROUP", "PRODUCT_COMBINATION", "CODE_REJECT_REASON"], ignore=["SK_ID_PREV"])
    p = p.join(a)
p = p.replace([np.inf, -np.inf], np.nan)
p.shape

(338857, 576)

# Save

In [9]:
p.reset_index().to_pickle(INTERIM / "previous_application.pkl")
p.shape

(338857, 576)